In [ ]:
import os
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

In [ ]:
DATA_DIR = Path(r"G:\rice-leaf-disease-xai\data\Rice Dataset")
CLASSES = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print("Detected classes:", CLASSES)
print("Total classes:", len(CLASSES))

In [ ]:
counts = {}
for cls in CLASSES:
    cls_dir = DATA_DIR / cls
    imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    counts[cls] = len(imgs)

df_counts = pd.DataFrame(list(counts.items()), columns=["class", "count"]).sort_values("count", ascending=False)
print(df_counts)
print("Total images:", df_counts["count"].sum())

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(df_counts["class"], df_counts["count"], color="seagreen")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Image count")
plt.title("Class distribution")
plt.tight_layout()
plt.show()

In [ ]:
max_c = df_counts["count"].max()
min_c = df_counts["count"].min()
print("Imbalance ratio (max/min):", round(max_c/min_c, 2))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16,8))
axes = axes.flatten()

for i, cls in enumerate(CLASSES):
    cls_dir = DATA_DIR / cls
    img_path = next(f for f in cls_dir.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"])
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis("off")

for j in range(len(CLASSES), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sizes = []
for cls in CLASSES:
    cls_dir = DATA_DIR / cls
    for f in cls_dir.iterdir():
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            try:
                with Image.open(f) as im:
                    sizes.append(im.size)
            except Exception as e:
                print("Error:", f, e)

sizes_arr = np.array(sizes)
print("Unique sizes:", len(set(sizes)))
print("Width  - min/max/mean:", sizes_arr[:,0].min(), sizes_arr[:,0].max(), sizes_arr[:,0].mean())
print("Height - min/max/mean:", sizes_arr[:,1].min(), sizes_arr[:,1].max(), sizes_arr[:,1].mean())

In [ ]:
corrupt = []
for cls in CLASSES:
    cls_dir = DATA_DIR / cls
    for f in cls_dir.iterdir():
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            try:
                img = Image.open(f)
                img.verify()
            except Exception:
                corrupt.append(str(f))

print("Corrupted images found:", len(corrupt))
if corrupt:
    print(corrupt)

In [ ]:
formats = Counter()
modes = Counter()
for cls in CLASSES:
    cls_dir = DATA_DIR / cls
    for f in cls_dir.iterdir():
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            with Image.open(f) as im:
                formats[im.format] += 1
                modes[im.mode] += 1

print("Formats:", formats)
print("Color modes:", modes)

In [ ]:
df_counts.to_csv("eda_class_distribution.csv", index=False)
print("Saved: eda_class_distribution.csv")